In [ ]:
!pip install -q transformers sentence-transformers razdel

# Генерация train_augmentation.csv

In [ ]:
import re
import random
import warnings
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

device: cuda


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'
output_dir = os.path.join(drive_root, 'output')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(drive_root, 'train.csv')
out_path = os.path.join(output_dir, 'paraphrase')

Mounted at /content/drive


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
import pandas as pd

df = pd.read_csv(train_path)
counts = df["label"].value_counts()
small_labels = counts[counts < 30].sort_values(ascending=True).index
df_small = df[df["label"].isin(small_labels)]

for label in small_labels:
    print("=" * 80)
    print(f"LABEL: {label}")
    print("=" * 80)

    texts = df_small.loc[df_small["label"] == label, "text"]
    for i, t in enumerate(texts, start=1):
        print(f"\n--- sample {i} ---\n")
        print(t)

    print("\n\n")

LABEL: Имущественные вопросы

--- sample 1 ---

[ORGANIZATION] ПРИКАЗ [DATE_TIME] No [DOCUMENT_NUMBER] Об утверждении Положений по непрофильным активам и порядке отчуждения непрофильных активов В целях приведения локальных нормативных актов Группы [ORGANIZATION] по вопросам управления непрофильными активами в соответствии с нормативными актами [ORGANIZATION] ПРИКАЗЫВАЮ: 1. Утвердить Положение о Комиссии по непрофильным активам [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 1). 2. Утвердить Положение о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 2). 3. Утвердить состав Комиссии по непрофильным активам [ORGANIZATION] в новом составе (Приложение 3). 4. Признать утратившими силу приказы [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] и от [DATE_TIME] No [DOCUMENT_NUMBER]. 5. Распространить действие настоящего Приказа на организации Группы компаний [ORGANIZATION]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from razdel import sentenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------- 1. Модель перефразирования (ruT5) --------
paraphrase_model_name = "fyaronskiy/ruT5-large-paraphraser"
para_tokenizer = AutoTokenizer.from_pretrained(paraphrase_model_name)
para_model = AutoModelForSeq2SeqLM.from_pretrained(paraphrase_model_name).to(device)

def generate_paraphrase(
    text,
    beams: int = 3,
    grams: int = 3,
    do_sample: bool = True,
    num_return_sequences: int = 1,
    top_k: int = 30,
    top_p: float = 0.95,
    temperature: float = 1.1,
):
    x = para_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    max_size = int(x.input_ids.shape[1] * 2.0)

    output = para_model.generate(
        **x,
        encoder_no_repeat_ngram_size=grams,
        no_repeat_ngram_size=3,
        num_beams=beams,
        max_length=max_size,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
    )
    return [para_tokenizer.decode(o, skip_special_tokens=True) for o in output]

# -------- 2. Модель эмбеддингов (deepvk/USER-base) --------
embed_model_name = "deepvk/USER-base"
embed_model = SentenceTransformer(embed_model_name, device=device)

# -------- 3. Один тестовый текст --------
source_text = "[ORGANIZATION] ПРИКАЗ [DATE_TIME] No [DOCUMENT_NUMBER] Об утверждении Положений по непрофильным активам и порядке отчуждения непрофильных активов В целях приведения локальных нормативных актов Группы [ORGANIZATION] по вопросам управления непрофильными активами в соответствии с нормативными актами [ORGANIZATION] ПРИКАЗЫВАЮ: 1. Утвердить Положение о Комиссии по непрофильным активам [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 1). 2. Утвердить Положение о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 2). 3. Утвердить состав Комиссии по непрофильным активам [ORGANIZATION] в новом составе (Приложение 3). 4. Признать утратившими силу приказы [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] и от [DATE_TIME] No [DOCUMENT_NUMBER]. 5. Распространить действие настоящего Приказа на организации Группы компаний [ORGANIZATION], в которых Компания имеет право прямо или косвенно распоряжаться более 50 голосов (Приложение 4). 6. Контроль исполнения приказа возложить на заместителя Председателя Правления [PERSON] Председатель Правления [PERSON] Приложение 1 к приказу [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] ПОЛОЖЕНИЕ О КОМИССИИ [ORGANIZATION] И ОРГАНИЗАЦИЙ ГРУППЫ КОМПАНИЙ [ORGANIZATION] ПО НЕПРОФИЛЬНЫМ АКТИВАМ 1. Общие положения Настоящее Положение определяет цели, задачи, организацию деятельности Комиссии [ORGANIZATION] по непрофильным активам (далее – Комиссия), права и обязанности Председателя, Секретаря и членов Комиссии, а также порядок представления отчетов о ходе ее работы. Требования настоящего Положения обязательны для применения при отчуждении непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION]. Дочерние общества и организации [ORGANIZATION] обеспечивают утверждение локальных нормативных актов (положений), регламентирующих в соответствии с настоящим Положением порядок работы комиссий по непрофильным активам в дочерних обществах, а также в объектах вложений дочерних обществ, являющихся организациями Группы компаний [ORGANIZATION]. Представители интересов [ORGANIZATION] и дочерних обществ [ORGANIZATION] в органах управления организаций Группы компаний [ORGANIZATION] обеспечивают на основе настоящего Положения утверждение локальных нормативных актов (положений), регламентирующих порядок работы комиссий по непрофильным активам в таких организациях Группы компаний [ORGANIZATION]. 2. Основные цели и задачи Комиссии 2.1. Комиссия создается в целях обеспечения реализации Стратегии по управлению имуществом и иными активами [ORGANIZATION], Программы отчуждения непрофильных активов [ORGANIZATION], утверждаемой [ORGANIZATION], и Положения о комиссии [ORGANIZATION] по непрофильным активам. Комиссия в своей деятельности руководствуется законодательством Российской Федерации, решениями органов управления [ORGANIZATION] и [ORGANIZATION], иными внутренними документами, в том числе Положением о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы Газпром, Положением о комиссии [ORGANIZATION] по непрофильным активам, Положением о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] и настоящим Положением. 2.2. Основной задачей Комиссии является принятие решений об отнесении активов, принадлежащих [ORGANIZATION] и организациям Группы компаний [ORGANIZATION], к категории непрофильных, а также определение порядка их реализации. В отношении активов, принадлежащих [ORGANIZATION] и организациям Группы компаний [ORGANIZATION], в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет 100 (сто) и более миллионов рублей с учетом НДС решения принимает Комиссия [ORGANIZATION] по непрофильным активам. В отношении активов, принадлежащих [ORGANIZATION], а также организациям Группы компаний [ORGANIZATION] в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет от 30 до 100 миллионов рублей с учетом НДС, решения принимаются Комиссией [ORGANIZATION]. В отношении активов, принадлежащих организациям Группы компаний [ORGANIZATION], в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет менее 30 миллионов рублей с учетом НДС, решения принимаются Комиссией организации Группы компаний [ORGANIZATION]. 2.3. Для выполнения возложенной на нее задачи, Комиссия определяет критерии отнесения активов к категории непрофильных, а также принимает следующие решения: об отнесении активов к категории непрофильных; о признании непрофильных активов подлежащими/не подлежащими отчуждению; о целесообразности применения тех или иных способов отчуждения непрофильных активов; об определении условий отчуждения непрофильных активов2; о необходимости проведения предпродажной подготовки непрофильных активов, в том числе оценки их рыночной стоимости. 3. Организация деятельности Комиссии 3.1. Персональный состав Комиссии утверждается приказом [ORGANIZATION]. 3.2. Вопросы об отнесении активов к категории непрофильных вносятся на рассмотрение Комиссии по инициативе органов управления [ORGANIZATION], заместителей Председателя Правления и руководителей подразделений прямого подчинения Председателю Правления [ORGANIZATION], структурных подразделений [ORGANIZATION]. 3.3. Организация Группы компаний [ORGANIZATION] перед представлением материалов на рассмотрение Комиссии согласовывает обоснование целесообразности отчуждения актива, а также предполагаемого способа его отчуждения со структурными подразделениями [ORGANIZATION], ответственными за осуществление контроля за обеспечением эффективности долгосрочных финансовых вложений [ORGANIZATION] в соответствующие дочерние общества [ORGANIZATION], которым подконтрольна эта организация Группы компаний [ORGANIZATION] (далее - ответственное подразделение [ORGANIZATION]). Представление материалов на рассмотрение Комиссии осуществляется дочерними обществами [ORGANIZATION], в том числе в отношении подконтрольных организаций Группы компаний [ORGANIZATION], являющихся их объектами вложений, либо ответственным подразделением [ORGANIZATION]. 3.4 Комиссия по вопросам, входящим в ее компетенцию, имеет право: запрашивать у структурных подразделений [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] необходимые для ее деятельности документы, материалы и информацию; устанавливать для структурных подразделений [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] сроки и форму представления 1 В отношении объектов имущества, имеющих одни и те же идентификационные признаки (место нахождения, назначение и другие признаки), а также отчуждаемых в пользу одного приобретателя либо входящих в единый имущественный комплекс, рассчитывается суммарная балансовая (остаточная) или рыночная или кадастровая стоимость имущества (для земельных участков). 2 Во избежание дублирования норм и правил работы Комиссии и Инвестиционных комитетов Компании, вопросы о целесообразности отчуждения непрофильных активов могут быть приняты к рассмотрению Комиссией только после их рассмотрения на соответствующем Инвестиционном Комитете запрашиваемых документов, материалов и информации. 3.5. Заседания Комиссии проводятся по мере необходимости. Заседание Комиссии является правомочным при участии в нем не менее половины от общего числа ее членов. При очной форме проведения заседания Комиссии решения принимаются открытым голосованием простым большинством голосов членов Комиссии, присутствующих на заседании. При заочной форме проведения заседания Комиссии голосование осуществляется посредством бюллетеней, которые направляются ее членам в соответствии с настоящим Положением. Бюллетени для голосования должны содержать указание на дату представления заполненного бюллетеня. В случае равенства голосов решающим является голос Председателя Комиссии, а в его отсутствие заместителя Председателя Комиссии. 3.6. Повестка заседания Комиссии, письмо о созыве заседания с указанием формы, даты, времени и места его проведения, подписанные Председателем Комиссии (в его отсутствие заместителем Председателя Комиссии), информационные материалы и бюллетени для голосования (при проведении заседания в заочной форме) направляются членам Комиссии за 5 календарных дней до даты проведения заседания Комиссии. В исключительных случаях при необходимости принятия оперативных решений указанный срок может быть сокращен по решению Председателя Комиссии (в его отсутствие заместителя Председателя Комиссии). 3.7. Итоги голосования членов Комиссии оформляются протоколом в течение 3 календарных дней с даты проведения заседания. Протокол подписывается Председателем Комиссии (в его отсутствие заместителем Председателя Комиссии) и Секретарем Комиссии. При принятии Комиссией решений заочным голосованием протокол заседания оформляется в течение 3 календарных дней с даты, установленной для представления заполненных бюллетеней. К протоколу прилагаются подписанные членами Комиссии бюллетени для голосования. Бюллетень члена Комиссии, не представленный Секретарю Комиссии в установленный для голосования срок, либо заполненный ненадлежащим образом, при подведении итогов голосования не учитывается. Члены Комиссии, голосовавшие против принятого решения, а также воздержавшиеся вправе в письменной форме изложить свое особое мнение, которое приобщается к протоколу заседания Комиссии. Протокол заседания Комиссии должен содержать информацию о членах Комиссии, проголосовавших по вопросам повестки заседания, результатах голосования, а также о принятых на заседании решениях. 3.8. Решение Комиссии (выписки из протокола заседания Комиссии) направляются для исполнения в соответствующее структурное подразделение или дочернее общество [ORGANIZATION] в течение 3 рабочих дней с даты оформления соответствующего протокола Комиссии. 4. Права и обязанности Председателя, Секретаря и членов Комиссии 4.1. Председателем Комиссии назначается руководитель структурного подразделения [ORGANIZATION], одной из основных задач которого является обеспечение управления непрофильными активами [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] (далее Департамент по правовым, корпоративным и имущественным вопросам). В отсутствие Председателя Комиссии обязанности Председателя исполняет его заместитель. Председатель Комиссии: организует работу Комиссии и выполнение возложенных на нее задач; созывает, проводит заседания Комиссии и председательствует на них; обеспечивает коллегиальное обсуждение рассматриваемых вопросов; при необходимости дает поручения членам Комиссии;"

print("Исходный текст:")
print(source_text)

# -------- 4. Разбиение на предложения через razdel --------
sentences = [s.text for s in sentenize(source_text)]
print("\nПредложения после разбиения:")
for i, s in enumerate(sentences, 1):
    print(f"[{i}] {s}")

# -------- 5. Перефразирование каждого предложения --------
paraphrased_sentences = []
for i, s in enumerate(sentences, 1):
    # можно добавить фильтры по длине, если нужно
    paras = generate_paraphrase(
        s,
        beams=3,
        grams=3,
        do_sample=True,
        num_return_sequences=1,
        top_k=50,
        top_p=0.95,
        temperature=1.1,
    )
    paraphrased = paras[0]
    paraphrased_sentences.append(paraphrased)
    print(f"\nИсходное предложение [{i}]: {s}")
    print(f"Перефраз [{i}]: {paraphrased}")

# -------- 6. Склейка предложений обратно в текст --------
paraphrased_text = " ".join(paraphrased_sentences)

print("\nСклеенный перефразированный текст:")
print(paraphrased_text)

# -------- 7. Косинусная схожесть оригинал ↔ перефраз --------
texts_for_embed = [source_text, paraphrased_text]
embeddings = embed_model.encode(texts_for_embed, convert_to_tensor=True)

src_emb, para_emb = embeddings[0], embeddings[1]
cos_score = util.cos_sim(src_emb, para_emb).item()

print(f"\nCosine similarity (оригинал ↔ перефразированный текст): {cos_score:.4f}")

Device: cuda


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Исходный текст:
[ORGANIZATION] ПРИКАЗ [DATE_TIME] No [DOCUMENT_NUMBER] Об утверждении Положений по непрофильным активам и порядке отчуждения непрофильных активов В целях приведения локальных нормативных актов Группы [ORGANIZATION] по вопросам управления непрофильными активами в соответствии с нормативными актами [ORGANIZATION] ПРИКАЗЫВАЮ: 1. Утвердить Положение о Комиссии по непрофильным активам [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 1). 2. Утвердить Положение о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 2). 3. Утвердить состав Комиссии по непрофильным активам [ORGANIZATION] в новом составе (Приложение 3). 4. Признать утратившими силу приказы [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] и от [DATE_TIME] No [DOCUMENT_NUMBER]. 5. Распространить действие настоящего Приказа на организации Группы компаний [ORGANIZATION], в которых Компания имеет право

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from razdel import sentenize

SIM_MIN = 0.70
SIM_MAX = 0.95

aug_rows = []

aug_file_path = os.path.join(out_path, "train_paraphrase_partial.csv")
if os.path.exists(aug_file_path):
    df_prev = pd.read_csv(aug_file_path)
    aug_rows = df_prev.to_dict(orient="records")
    print(f"Загружено уже сгенерированных примеров: {len(aug_rows)}")

label_to_texts = {
    label: df_small.loc[df_small["label"] == label, "text"].tolist()
    for label in small_labels
}

for label in tqdm(small_labels, desc="Labels"):
    texts_orig = label_to_texts[label]
    current_count = len(texts_orig)
    already_aug = [r for r in aug_rows if r["label"] == label]
    current_count_with_aug = current_count + len(already_aug)

    if current_count_with_aug >= 30:
        continue

    need = 30 - current_count_with_aug
    print(f"\nLabel: {label} | есть {current_count_with_aug}, нужно добить: {need}")

    orig_embeddings = embed_model.encode(texts_orig, convert_to_tensor=True)

    orig_idx = 0
    attempts = 0
    max_attempts = need * 20

    while need > 0 and attempts < max_attempts:
        attempts += 1
        text_orig = texts_orig[orig_idx]
        orig_emb = orig_embeddings[orig_idx]

        # --- разбиение на предложения ---
        sentences = [s.text for s in sentenize(text_orig)]

        # --- перефраз каждого предложения и склейка ---
        paraphrased_sentences = []
        for s in sentences:
            # можно добавить фильтр по длине, если нужно
            paras = generate_paraphrase(
                s,
                beams=3,
                grams=3,
                do_sample=True,
                num_return_sequences=1,
                top_k=30,
                top_p=0.95,
                temperature=1.1,
            )
            paraphrased_sentences.append(paras[0])

        para_text = " ".join(paraphrased_sentences)

        # --- cosine similarity оригинал ↔ перефразированный документ ---
        emb_para = embed_model.encode(para_text, convert_to_tensor=True)
        sim = util.cos_sim(orig_emb, emb_para).item()

        if SIM_MIN <= sim <= SIM_MAX:
            aug_rows.append({
                "label": label,
                "text": para_text,
                "source_text": text_orig,
                "cosine_sim": sim,
            })
            need -= 1

            if len(aug_rows) % 10 == 0:
                df_aug_partial = pd.DataFrame(aug_rows)
                df_aug_partial.to_csv(aug_file_path, index=False)
                print(f"Сохранено {len(aug_rows)} аугментированных примеров в {aug_file_path}")

        orig_idx = (orig_idx + 1) % len(texts_orig)

# финальное сохранение аугментаций
df_aug = pd.DataFrame(aug_rows)
df_aug.to_csv(aug_file_path, index=False)
print(f"\nИтого аугментированных примеров: {len(df_aug)}")
print(f"Промежуточный файл сохранён в: {aug_file_path}")

# склейка с исходным df
df_full = pd.concat([df, df_aug[["label", "text"]]], ignore_index=True)
final_path = os.path.join(out_path, "train_paraphrase.csv")
df_full.to_csv(final_path, index=False)
print(f"Финальный датасет сохранён в: {final_path}")

Labels:   0%|          | 0/23 [00:00<?, ?it/s]


Label: Имущественные вопросы | есть 1, нужно добить: 29
Сохранено 10 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphrase_partial.csv
Сохранено 20 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphrase_partial.csv

Label: Проект «Трубопроводный транспорт Ещё одного НГКМ» | есть 1, нужно добить: 29
Сохранено 30 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphrase_partial.csv
Сохранено 40 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphrase_partial.csv
Сохранено 50 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphrase_partial.csv

Label: Блок заместителя генерального директора по строительству | есть 1, нужно добить: 29
Сохранено 60 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/paraphrase/train_paraphras